## 1. Run KEGG/KOfam Annotation


In [ ]:
%%bash
# Run the annotation program for each extracted genome file.
set -euo pipefail

EXTRACT_DIR=/home/daisy/data/try/download_test/extract

for input_file in "${EXTRACT_DIR}"/*; do
    [ -f "$input_file" ] || continue
    filename=$(basename "$input_file")
    prefix=${filename:0:15}
    exec_annotation -o "${prefix}.out" "$input_file"
done


## 2. Extract KO Identifiers from Annotation Results


In [ ]:
%%bash
# Extract KO identifiers for each annotated genome.
set -euo pipefail

ACCESSION_FILE=/home/daisy/data/try/download_test/accession_number.linux
OUTPUT_DIR=/home/daisy/data/try/download_test/KEGG_annotation_Knumber
mkdir -p "$OUTPUT_DIR"

for ((i=2; i<=242; i++)); do
    accession_number=$(sed -n "${i}p" "$ACCESSION_FILE")
    grep -F '*' "${accession_number}.out" | awk '{print $3}' \
        > "${OUTPUT_DIR}/${accession_number}_Knumber"
done


## 3. Build the Strain-by-KO Count Matrix


In [ ]:
from pandas import DataFrame
import pandas as pd

def makeMatrix(StrainList, ListFile, suffix, ResultFile):
    #StrainList是对应的菌种名的txt文件
    #ListFile是K号文件对应的文件夹
    #suffix是K号文件的后缀名
    #ResultFile是输出结果的文件
    newcontents = []
    knumber_contents = []

    filename = StrainList
    # 读取accession number/菌株名到列表里
    #content表示StrainList文件里的每一行的菌株名
    #newcontents则是这些菌株名组成的list
    with open(filename, encoding="UTF-8") as f:
        contents = f.readlines()

    for content in contents:
        content = content.replace('\n', '')
        newcontents.append(content)

    # 根据accession提取文件，把文件里的Knumber拼到一个list（knumber_contents）里，并且把每个基因组的Knumber做成一个列表


    file_list = []         #创建空列表,用以储存每一个菌株名及其对应的Knumber
    line_count=len(newcontents)
    for i in range(line_count):
        file_list.append([])
    i=0
    for newcontent in newcontents:

        filename = ListFile+ "/"+newcontent+ suffix
        file_list[i].append(newcontent)

        with open(filename, encoding="UTF-8") as f:
            contents = f.readlines()

        for content in contents:
            content = content.replace('\n', '')
            knumber_contents.append(content)
            file_list[i].append(content)

        i=i+1

    only_knumbers = []
    for el in knumber_contents:
        if only_knumbers.count(el) < 1:
            only_knumbers.append(el)

    #准备工作完成，开始制作count次数的嵌套字典


    number1 = {}
    i = 0
    while i <= line_count - 1:
        b = {}
        for only_knumber in only_knumbers:
            b[only_knumber] = file_list[i].count(only_knumber)

        number1[file_list[i][0]] = b #file_list[i][0]表示每个菌株列表的第1个元素，也就是菌株名
        i = i + 1

    df = DataFrame(number1)
    dfT = df.T
    print(df)
    dfT.to_csv(ResultFile)